In [1]:
%load_ext autoreload
%autoreload 2

from hw1_imitation.model import build_model

In [2]:
import torch
from torch.utils.data import DataLoader

from hw1_imitation.data import (
    Normalizer,
    PushtChunkDataset,
    download_pusht,
    load_pusht_zarr,
)
from hw1_imitation.model import build_policy
from hw1_imitation.train import parse_train_config

In [24]:
config = parse_train_config(args=[])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

zarr_path = download_pusht(config.data_dir)
states, actions, episode_ends = load_pusht_zarr(zarr_path)
normalizer = Normalizer.from_data(states, actions)

In [33]:
# Test out data loading and develop a training loop



dataset = PushtChunkDataset(
    states,
    actions,
    episode_ends,
    chunk_size=config.chunk_size,
    normalizer=normalizer,
)

loader = DataLoader(
    dataset,
    batch_size=config.batch_size,
    shuffle=True,
    drop_last=True,
)

model = build_policy(
    "flow", # Use the flow matching policy class instead
    state_dim=states.shape[1],
    action_dim=actions.shape[1],
    chunk_size=config.chunk_size,
    hidden_dims=config.hidden_dims,
).to(device)

In [34]:
model

FlowMatchingPolicy(
  (policy): Sequential(
    (0): Linear(in_features=5, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=1024, bias=True)
    (3): ReLU()
    (4): Linear(in_features=1024, out_features=512, bias=True)
    (5): ReLU()
    (6): Linear(in_features=512, out_features=512, bias=True)
    (7): ReLU()
    (8): Linear(in_features=512, out_features=16, bias=True)
  )
)

In [35]:
for batch in loader:
    batch = [b.to(device) for b in batch]
    break

In [36]:
print(len(batch))
print(batch[0].type()) # State, [128, 5], which is batch_size, state_dim
print(batch[0].shape)

print(batch[1].type()) # Action chunk, [128, 8, 2], which is batch_size, chunk_size, action_dim
print(batch[1].shape)

2
torch.cuda.FloatTensor
torch.Size([128, 5])
torch.cuda.FloatTensor
torch.Size([128, 8, 2])


In [ ]:
# Look at hte output
state = batch[0]
sample_pred = model.compute_loss(state, batch[1]) # 128, 8, 2] 128 is the batch size, 8 is the chunk size, and 2 is the action dimension (x,y)
print(sample_pred.shape)
print(sample_pred[0])  # 8, 2] 8 is the chunk size, and 2 is the action dimension (x,y)

Batch size: 128
torch.Size([128, 8, 2])
tensor([[ 0.0281,  0.0187],
        [-0.0114,  0.0358],
        [ 0.0185,  0.0008],
        [-0.0062,  0.0160],
        [ 0.0208, -0.0729],
        [-0.0204, -0.0677],
        [ 0.0008,  0.0156],
        [ 0.0342,  0.0251]], device='cuda:0', grad_fn=<SelectBackward0>)


In [38]:
config.batch_size

128

In [6]:
len(loader)

189

In [ ]:

# develop a training loop

# Batch has two elements.  State and action.
# The actions are [128, 8, 2] 128 is the batch size, 8 is the chunk size, and 2 is the action dimension (x,y)
# Chunk size is how many actions we are predicting at once with the policy.  
# Given a state, we predict the next 8 actions.  
# This means the model itself needs to output the two x,y coordinates 8 times.  
# The nn.Linear layer will just return 16 rows, then we can reshape it to [8, 2]

# Does the original dataset have states for all the 8 subsetquent actions? 
# For the state, it is a 5 dimensional vector.  The first two are the x,y position of the end effector, the next two are the x,y position of the object, and the last is the angle of the object.  The model will take this 5 dimensional vector and output 16 values, which we will reshape to [8, 2] for the actions.

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
counter = 1
EPOCHS = 50

for epoch in range(EPOCHS):

    total_loss = 0.0
    for batch in loader:
        batch = [b.to(device) for b in batch]

        optimizer.zero_grad()

        # The compute loss function takes in the state and does the forward pass
        # print("Targets shape:", batch[1].shape)
        loss = model.compute_loss(batch[0], batch[1]) # state, action_chunk
        # print("loss:", loss)
        loss.backward()
        optimizer.step()

        # Backward Pass

        # Reset the gradients
        counter += 1
        total_loss += loss.item()
    epoch_loss = total_loss / len(loader)
    print(f"Epoch {epoch} done.  Loss: {epoch_loss}")


Epoch 0 done.  Loss: 0.17895988903190724
Epoch 1 done.  Loss: 0.11564420814078952
Epoch 2 done.  Loss: 0.10141686223959796
Epoch 3 done.  Loss: 0.09341961445001068
Epoch 4 done.  Loss: 0.08620617004535186
Epoch 5 done.  Loss: 0.08103545542274203
Epoch 6 done.  Loss: 0.07652961971267822
Epoch 7 done.  Loss: 0.0719763453281115
Epoch 8 done.  Loss: 0.06715828371505259
Epoch 9 done.  Loss: 0.06410539572043393
Epoch 10 done.  Loss: 0.06180706113655731
Epoch 11 done.  Loss: 0.059356171302694494
Epoch 12 done.  Loss: 0.05632842966803798
Epoch 13 done.  Loss: 0.054521947941452105
Epoch 14 done.  Loss: 0.0534763955092304
Epoch 15 done.  Loss: 0.05200753593570972
Epoch 16 done.  Loss: 0.049883561553778474
Epoch 17 done.  Loss: 0.048447688864100544
Epoch 18 done.  Loss: 0.046846983520678745
Epoch 19 done.  Loss: 0.04598098656251317
Epoch 20 done.  Loss: 0.04449835450206169
Epoch 21 done.  Loss: 0.04341606912079942
Epoch 22 done.  Loss: 0.042716510662878
Epoch 23 done.  Loss: 0.041902237458440364


torch.Size([128, 8, 2])
tensor([[-1.4759,  0.6134],
        [-1.5126,  0.8382],
        [-1.4811,  1.0341],
        [-1.4185,  1.2424],
        [-1.3206,  1.4214],
        [-1.1936,  1.5683],
        [-1.0877,  1.6947],
        [-0.9852,  1.7778]], device='cuda:0', grad_fn=<SelectBackward0>)


In [24]:
from torch import nn

ultrasimple =nn.Linear(5, 10)
ultrasimple(batch[0]).shape

torch.Size([128, 10])

In [20]:
print(model)

MSEPolicy(
  (policy): Sequential(
    (0): Linear(in_features=5, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=256, bias=True)
    (3): ReLU()
    (4): Linear(in_features=256, out_features=256, bias=True)
    (5): ReLU()
    (6): Linear(in_features=256, out_features=2, bias=True)
  )
)


In [25]:
model(batch[0]).shape

NotImplementedError: Module [MSEPolicy] is missing the required "forward" function